# FER-2013: M1 custom CNN and M2 DDAMFN

This is the lab's local training and evaluation notebook. Read the architecture
and training explanations alongside the small helper functions in `src/`.
The implementation follows the lab PDF; M1 follows the later supplied image.

Select the `C:\Users\ADMIN\ai_venv` Jupyter kernel, install requirements,
and extract [FER-2013](https://www.kaggle.com/datasets/msambare/fer2013) before running.

In [ ]:
import sys
from pathlib import Path

# Works whether Jupyter starts in the repository root or in notebooks/.
candidates = [Path.cwd().resolve(), Path.cwd().resolve().parent]
PROJECT_ROOT = next((p for p in candidates if (p / "src" / "model.py").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Start Jupyter in this repository or its notebooks/ directory.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print("Project:", PROJECT_ROOT)
print("Python:", sys.executable)

In [ ]:
import json
import shutil
import torch
from IPython.display import Image, display

from src.data import CLASS_NAMES, build_data, preprocessing_for
from src.model import build_model, count_parameters, model_architecture
from src.ddamfn import download_pretrained, PRETRAINED_SHA256
from dataclasses import asdict
from src.train import TrainConfig, fit, seed_everything
from src.experiments import validate_checkpoint
from src.evaluate import plot_class_distribution, evaluate_checkpoint, compare_models
from src.report import build_report

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
DATA_ROOT = PROJECT_ROOT / "data" / "fer2013"  # Contains train/ and test/.
RUN_NAME = "fer2013_ddamfn"  # Change before starting another fresh run.
RUN_ROOT = PROJECT_ROOT / "runs" / RUN_NAME
OUTPUT_ROOT = RUN_ROOT / "outputs"
WEIGHTS_DIR = RUN_ROOT / "weights"
REPORT_PATH = RUN_ROOT / "report.pdf"
BATCH_SIZE = 128  # Matches the first Kaggle run; reduce if memory is insufficient.
NUM_WORKERS = 0
VAL_FRACTION = 0.10
SEED = 42

## 1. Inspect the data and the split

Both models use the same stratified train/validation split. Validation is held
out of the original `train/` directory, never from `test/`. The class weights
are computed **only from training labels**: square-root inverse frequency,
normalized to mean one. This gives rare classes more influence without the
very large weights of full inverse frequency.

FER images remain grayscale. M1 uses 1x48x48 and mean/std 0.5.
M2 repeats grayscale into three channels, resizes to 112x112, and uses the
pretrained mean (0.485, 0.456, 0.406) and std (0.229, 0.224, 0.225).
Upsampling adds no new detail; it preserves the pretrained model's input contract.
The app selects preprocessing from checkpoint metadata.

Each model rebuilds its training loader with the augmentation in its config.
Both use the same split seed and validation images. Loss weights remain
square-root inverse-frequency; oversampling is not added.

In [ ]:
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
seed_everything(SEED)
data = build_data(DATA_ROOT, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
                  val_fraction=VAL_FRACTION, seed=SEED)
counts = plot_class_distribution(data, OUTPUT_ROOT / "class_distribution.png")
data.save_manifest(OUTPUT_ROOT / "split_manifest.json")
print("Resolved data directory:", data.root)
print(json.dumps(counts, indent=2))
print("Class order:", CLASS_NAMES)
print("Training class weights:", data.class_weights.tolist())
display(Image(filename=str(OUTPUT_ROOT / "class_distribution.png")))

## 2. Inspect M1 and M2

**M1 matches the supplied architecture.** Each block contains one padded 3x3
convolution, BatchNorm, ReLU, 2x2 max pool, and ordinary dropout. Channels:
32, 64, 128, 256; dropout: 0.10, 0.15, 0.20, 0.25. Its spatial sizes are
24, 12, 6, 3, so flattening produces 2304 features. The head is
Linear(2304,256), BatchNorm, ReLU, Dropout(0.50), Linear(256,7).

**M2 is DDAMFN**, initialized from the official **DDAMFN++ AffectNet-7**
checkpoint. Mixed-kernel depthwise blocks learn face features; two coordinate
attention heads emphasize expression-related regions. The model has 4,187,421
parameters. We retain its 112x112, three-channel input and seven-emotion head,
remapping classifier rows to this project's alphabetical label order.
All layers are fine-tuned on the FER training split. This uses expression-labeled
external pretraining; record that when comparing against the earlier ImageNet ResNet.
The checkpoint revision and SHA-256 are pinned in `src/ddamfn.py`.
We use weighted cross entropy here, rather than reproducing upstream SAM or
its reciprocal attention-diversity loss.

Both networks return **logits**, not softmax, because CrossEntropyLoss applies
log-softmax itself. Inference applies softmax once to display confidence.

In [ ]:
torch.set_num_threads(4)
for name in ["cnn", "transfer"]:
    # Architecture inspection does not download pretrained weights.
    model = build_model(name, pretrained=False).eval()
    with torch.inference_mode():
        spec = preprocessing_for(model_architecture(name))
        scores = model(torch.zeros(2, spec["channels"], spec["image_size"], spec["image_size"]))
    print(f"{name}: {count_parameters(model):,} parameters; output {tuple(scores.shape)}")
    del model

## 3. Final training configuration: one recipe per model

This notebook performs exactly **two fits**: M1 once and M2 once. It needs
the cloned source and FER-2013 dataset, with no previous run outputs or
BASELINE_RESULTS_ROOT setting. Each fit keeps its best-validation-accuracy
checkpoint, with macro F1 breaking exact ties.

M1 uses the exact supplied CNN and gentle combined augmentation: flip,
reflect-padded 2-pixel crop, rotation up to 8 degrees, scale 0.95-1.05, and
brightness/contrast jitter 0.08 with probability 0.5. No extra dropout or erasing.

M2 uses `fer_moderate`: flip, rotation up to 10 degrees, shifts up to 5%,
scale 0.95-1.05, brightness/contrast jitter 0.15 with probability 0.7, and
erasing with probability 0.15 covering 1-4% of the image. Augmentation starts
at the original 48x48 resolution before resizing. No vertical flips or large
crops: removing eyes or mouths can make emotion labels unlearnable.

M2 uses backbone LR 1e-5, head LR 1e-4, weight decay 1e-4, smoothing 0.02,
a two-epoch backbone LR ramp, and a 40-epoch cosine schedule (patience 10).
Its batch size is capped at 64 to allow for the larger input and deeper backbone.
M1 keeps its previous recipe: up to 80 epochs, smoothing 0.05, batch size 128.
Both retain training-only square-root class weights; no oversampling is stacked.
These settings need validation on a full FER run; improved accuracy is not guaranteed.
Validation/test/inference use no random augmentation.

Training logs print one summary per epoch. Each fit starts fresh; rerunning a
training cell overwrites that model's files in OUTPUT_ROOT. Change RUN_NAME to
preserve another run. Best/last files do not contain optimizer-resume state.
Optional multi-candidate search remains available in src/experiments.py,
but this notebook does not call it.

In [ ]:
# One fixed recipe per model. No candidate search or archived results required.
cnn_config = TrainConfig(
    model_name="cnn", architecture="cnn", epochs=80, learning_rate=1e-3,
    weight_decay=1e-4, patience=18, label_smoothing=0.05,
    augmentation="gentle_combined", use_class_weights=True,
    pretrained=False, seed=SEED, device="auto",
)
transfer_config = TrainConfig(
    model_name="transfer", architecture="ddamfn", epochs=40,
    learning_rate=1e-4, backbone_lr=1e-5, warmup_epochs=2,
    weight_decay=1e-4, patience=10, label_smoothing=0.02,
    augmentation="fer_moderate", use_class_weights=True,
    pretrained=True, seed=SEED, device="auto",
)
# Fail early if Internet/download is unavailable, before spending time training M1.
if transfer_config.pretrained:
    transfer_config.pretrained_path = str(download_pretrained())
    print("Verified official DDAMFN++ AffectNet-7 weights, SHA-256:", PRETRAINED_SHA256)
print("M1:", asdict(cnn_config))
print("M2:", asdict(transfer_config))

## 4. Train M1 once from scratch

Build the chosen training augmentation and save the best validation checkpoint.

In [ ]:
data = build_data(
    DATA_ROOT, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
    val_fraction=VAL_FRACTION, seed=SEED, augmentation=cnn_config.augmentation,
    architecture=cnn_config.architecture,
)
cnn_run = fit(cnn_config, data, OUTPUT_ROOT / "cnn")
cnn_validation = validate_checkpoint(cnn_run["checkpoint"], data, OUTPUT_ROOT / "cnn")
print("M1 checkpoint:", cnn_run["checkpoint"])
print("M1 validation:", cnn_validation)

## 5. Fine-tune M2 once

Rebuild the training loader with M2 augmentation, retaining the same validation split.

In [ ]:
data = build_data(
    DATA_ROOT, batch_size=min(BATCH_SIZE, 64), num_workers=NUM_WORKERS,
    val_fraction=VAL_FRACTION, seed=SEED, augmentation=transfer_config.augmentation,
    architecture=transfer_config.architecture,
)
transfer_run = fit(transfer_config, data, OUTPUT_ROOT / "transfer")
transfer_validation = validate_checkpoint(
    transfer_run["checkpoint"], data, OUTPUT_ROOT / "transfer",
)
print("M2 checkpoint:", transfer_run["checkpoint"])
print("M2 validation:", transfer_validation)

## 6. Final test evaluation and M3 comparison

Evaluate each model's best validation checkpoint after its fixed training run finishes. Export accuracy,
macro F1, parameter counts, and CPU forward FPS, along with the full confusion
matrix and per-class report. CPU timing is **model-only**, batch one, float32,
four threads; it excludes face detection, preprocessing and camera I/O.
The application's displayed FPS measures capture + processing separately.

The lab targets (60% M1, 65% M2) are checked against measured test accuracy;
the code does not assume they have been achieved.

In [ ]:
all_metrics = []
for name in ["cnn", "transfer"]:
    architecture = "cnn" if name == "cnn" else "ddamfn"
    # Same split/images, but each architecture needs its own deterministic input transform.
    evaluation_data = build_data(
        DATA_ROOT, batch_size=min(BATCH_SIZE, 64), num_workers=NUM_WORKERS,
        val_fraction=VAL_FRACTION, seed=SEED, architecture=architecture,
    )
    metrics = evaluate_checkpoint(
        OUTPUT_ROOT / name / "best.pt", evaluation_data, device="auto",
    )
    all_metrics.append(metrics)
    shutil.copy2(OUTPUT_ROOT / name / "best.pt", WEIGHTS_DIR / f"{name}_best.pt")
    print(json.dumps(metrics, indent=2))

comparison = compare_models(all_metrics, OUTPUT_ROOT)
print(f"{'Model':<10} {'Accuracy':>10} {'Macro F1':>10} {'Parameters':>12} {'CPU FPS':>10}")
for row in comparison:
    print(f"{row['model']:<10} {row['test_accuracy']:>10.2%} "
          f"{row['test_macro_f1']:>10.4f} {row['parameters']:>12,} {row['cpu_model_fps']:>10.1f}")

## 7. M4 and M5: inspect plots and mistakes

Each model exports up to ten errors, covering different true classes when
possible. The probability-based comments in `error_analysis.md` are a starting
point: inspect the faces and add image-specific observations about pose,
occlusion, lighting or ambiguous expressions. They are not invented visual
explanations. If fewer than ten mistakes exist, all available mistakes are shown.

In [ ]:
for name in ["cnn", "transfer"]:
    print("\n", name.upper())
    for filename in ["training_curves.png", "confusion_matrix.png", "misclassified.png"]:
        path = OUTPUT_ROOT / name / filename
        if path.exists():
            display(Image(filename=str(path)))
    print((OUTPUT_ROOT / name / "error_analysis.md").read_text(encoding="utf-8"))

## 8. Generate the six-page report

This uses actual saved results. Until you provide a phone-camera screenshot
and finish your visual error analysis, treat the PDF as a draft. Run the app
locally and press **s** to save `outputs/demo_screenshot.jpg`, then regenerate
the report. Edit `src/report.py` to incorporate your own written analysis.

In [ ]:
screenshot = OUTPUT_ROOT / "demo_screenshot.jpg"
report_path = build_report(
    OUTPUT_ROOT, REPORT_PATH,
    screenshot=screenshot if screenshot.exists() else None,
    author="YOUR NAME / STUDENT ID",
)
print("Report:", report_path)
print("Weights:", WEIGHTS_DIR)

## Before submission

Keep all full FER training/evaluation outputs in this notebook. The PDF asks
for an **executed** `notebooks/train.ipynb`; running only the Kaggle notebook
does not execute this local file. If you train on Kaggle, also upload
`notebooks/train.ipynb` as a Kaggle notebook and run it against the same clone
(adjust its setup/data/output paths), or run the local notebook with a GPU.

Copy the best checkpoints into `weights/`, finish the report, and test:

```text
python src/app.py --source http://<ip>:8080/video --model cnn
python src/app.py --source test.jpg --model transfer
```

Record actual CPU realtime FPS on your laptop, with the phone connected to the
same Wi-Fi network. Source URLs and checkpoint paths live in `config.yaml`.

This run keeps its checkpoints in RUN_ROOT / "weights". After reviewing the results,
copy the two *_best.pt files into the project-level weights/ folder used by config.yaml.
Run the app locally, save the phone demo screenshot, and regenerate the report.